In [1]:
import ast
from langchain_openai import ChatOpenAI

import sys
import os

# 获取当前 notebook 所在目录的上层目录（即项目根目录）
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

# 添加到 Python 模块搜索路径
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from utils.database import Database

db = Database()


✅ 已加载平台配置: D:\1Grad\Code\wkf\src\scsagent\config\.env.windows


In [ ]:
tool = 'scvi-tools'
sql1 = "select id, llm_summary from docs where tool_id = (select id from tools where name=%s)"
db_res1 = db.execute_query(sql1, tool)
summaries = ["id: " + str(row[0]) + "\tsummary: " + row[1] for row in db_res1]
doc_summaries = "\n\n\n".join(summaries)

llm = ChatOpenAI(
    # model='deepseek-v3-hs',
    # model='deepseek-r1-hs',
    # model="qwen-plus",
    model="qwen3-max",
    # model="qwen3-coder-plus",
    base_url="",
    api_key="",
    
)

query = "使用scvi-tools进行differential expression任务，数据为/workspace/input/mouse.h5ad"
res = llm.invoke(
    f"""# 任务要求
将如下文档简介，按照完成用户需求的合适程度进行打分，并按照分数由大到小排序，返回其id组成的tuple，如(3, 1)。不要输出解释或者```等格式。
# 输入：
## 用户需求：
{query}
## 文档简介：
{doc_summaries}"""
).content
summaries_ids = ast.literal_eval(res)
summaries_ids = (
    (summaries_ids,) if isinstance(summaries_ids, int) else summaries_ids
)
placeholders = ",".join(["%s"] * len(summaries_ids))
print("doc_ids: ", str(summaries_ids))

In [2]:
import numpy as np
import faiss
from typing import List

# ========== 1. 数据库查询 ==========
tool = 'scvi-tools'
sql1 = "SELECT id, doc FROM docs WHERE tool_id = (SELECT id FROM tools WHERE name=%s)"
db_res1 = db.execute_query(sql1, tool)

doc_ids = [row[0] for row in db_res1]      # 文档 ID 列表
doc_texts = [row[1] for row in db_res1]    # 文档内容列表

print(f"✅ 加载 {len(doc_texts)} 篇文档")

# ========== 2. 自定义 Embedding ==========
from openai import OpenAI

class CustomOpenAIEmbeddings:
    def __init__(self, api_key: str, base_url: str, model: str):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """为多个文档生成 embedding（逐个调用，适配不支持批量的本地服务）"""
        embeddings = []
        for i, text in enumerate(texts):
            try:
                # 确保 text 是字符串，避免传入 None 或非 str
                input_text = str(text) if text is not None else ""
                response = self.client.embeddings.create(
                    model=self.model,
                    input=input_text
                )
                embedding = response.data[0].embedding
                embeddings.append(embedding)
            except Exception as e:
                print(f"⚠️ Embedding failed for document {i}: {e}")
                # 可选：填充零向量或跳过
                raise  # 或者根据需求处理
        return embeddings

    def embed_query(self, text: str) -> List[float]:
        """为查询文本生成 embedding"""
        input_text = str(text) if text is not None else ""
        response = self.client.embeddings.create(
            model=self.model,
            input=input_text
        )
        return response.data[0].embedding
    
# 初始化 embedding
api_key = "sk-kpsteSkpDGl1xBmDEcC7D51b968e43499092826f17286b55"
base_url = "http://10.224.28.80:3000/v1"
model = "text-embedding-v4"
embedding_fn = CustomOpenAIEmbeddings(api_key=api_key, base_url=base_url, model=model)

# ========== 3. 用户查询 ==========
query = "使用scvi-tools进行differential expression任务，数据为/workspace/input/mouse.h5ad"
query_vector = np.array(embedding_fn.embed_query(query)).astype('float32')

# ========== 4. 文档向量化 ==========
print("正在生成文档向量...")
doc_vectors = np.array(embedding_fn.embed_documents(doc_texts)).astype('float32')

# ========== 5. 构建 FAISS 索引（使用余弦相似度） ==========

# FAISS 默认使用内积（用于余弦相似度需先归一化）
dimension = doc_vectors.shape[1]

# 👉 关键：归一化向量，使内积 = 余弦相似度
faiss.normalize_L2(doc_vectors)   # 归一化文档向量
faiss.normalize_L2(query_vector.reshape(1, -1))  # 归一化查询向量（可选，FAISS搜索时自动处理）

# 创建索引
index = faiss.IndexFlatIP(dimension)  # Inner Product = Cosine Similarity（归一化后）
index.add(doc_vectors)

# ========== 6. 执行搜索 ==========
k = len(doc_ids)  # 返回所有文档的排序
scores, indices = index.search(query_vector.reshape(1, -1), k)

# scores 是内积（=余弦相似度，因为已归一化），越大越相似
# indices 是 doc_vectors 中的索引位置

# ========== 7. 映射回原始 doc_id 并排序 ==========
sorted_doc_ids = [doc_ids[i] for i in indices[0]]

print("✅ 按余弦相似度排序的文档 ID：")
print(sorted_doc_ids)

# ========== 8. （可选）同时返回相似度分数 ==========
sorted_scores = scores[0].tolist()
id_score_pairs = list(zip(sorted_doc_ids, sorted_scores))
print("\n📄 ID + 相似度分数：")
for doc_id, score in id_score_pairs:
    print(f"ID: {doc_id}, Similarity: {score:.4f}")

✅ 加载 7 篇文档
正在生成文档向量...
✅ 按余弦相似度排序的文档 ID：
[1172, 1173, 1167, 1168, 1170, 1171, 1169]

📄 ID + 相似度分数：
ID: 1172, Similarity: 0.7313
ID: 1173, Similarity: 0.6990
ID: 1167, Similarity: 0.6624
ID: 1168, Similarity: 0.6545
ID: 1170, Similarity: 0.6453
ID: 1171, Similarity: 0.6261
ID: 1169, Similarity: 0.5984
